# Linear Regression

California housing dataset. Target: `median_house_value`.

## Data inspection

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn as sk
from sklearn.model_selection import train_test_split
from sklearn import linear_model

data_df = pd.read_csv("housing.csv")

print("Shape:", data_df.shape)
print("\nDtypes:\n", data_df.dtypes)
print("\nNulls:\n", data_df.isnull().sum())
data_df.head(10)

In [ ]:
data_df.describe()

## Feature vs target plots

In [ ]:
numerical_features = data_df.select_dtypes(include=[np.number]).columns.drop('median_house_value')

fig, axs = plt.subplots(4, 2, figsize=(20, 24))
fig.suptitle("Feature vs Median House Value", fontsize=24)

for idx, feature in enumerate(numerical_features):
    row, col = idx // 2, idx % 2
    axs[row, col].scatter(data_df[feature], data_df['median_house_value'], alpha=0.1, s=5)
    axs[row, col].set_xlabel(feature, fontsize=14)
    axs[row, col].set_ylabel('median_house_value', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
scatter = ax.scatter(data_df['longitude'], data_df['latitude'],
                     c=data_df['median_house_value'], cmap='magma',
                     alpha=0.4, s=10)
plt.colorbar(scatter, label='Median House Value ($)')
ax.set_xlabel('Longitude', fontsize=14)
ax.set_ylabel('Latitude', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(18, 6))

data_df['ocean_proximity'].value_counts().plot(kind='bar', ax=axs[0], color='steelblue')
axs[0].set_ylabel('Count', fontsize=14)
axs[0].tick_params(axis='x', rotation=45)

data_df.boxplot(column='median_house_value', by='ocean_proximity', ax=axs[1])
axs[1].set_ylabel('Median House Value ($)', fontsize=14)
axs[1].set_xlabel('Ocean Proximity', fontsize=14)
plt.suptitle('')

plt.tight_layout()
plt.show()

## Preprocessing

Fill null `total_bedrooms` with median, one-hot encode `ocean_proximity`, z-score standardize, add bias column.

In [ ]:
data_df['total_bedrooms'] = data_df['total_bedrooms'].fillna(data_df['total_bedrooms'].median())

data_encoded = pd.get_dummies(data_df, columns=['ocean_proximity'], dtype=float)

print("Encoded shape:", data_encoded.shape)
data_encoded.head()

In [ ]:
data = data_encoded.to_numpy()
target_col = data_encoded.columns.get_loc('median_house_value')

feature_cols = [i for i in range(data.shape[1]) if i != target_col]
x_data = data[:, feature_cols]
y = data[:, target_col]

feature_names = [data_encoded.columns[i] for i in feature_cols]

mu = x_data.mean(axis=0)
sigma = x_data.std(axis=0)
sigma[sigma == 0] = 1

r_data = (x_data - mu) / sigma
r_data = np.c_[np.ones(r_data.shape[0]), r_data]
print(f"r_data shape: {r_data.shape}")

## Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    r_data, y, test_size=0.33, shuffle=True, random_state=42
)

print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

## Gradient descent vs normal equation

GD: $\theta \leftarrow \theta - \alpha \cdot \frac{2}{n} X^T(X\theta - y)$

Normal eq: $\hat\theta = (X^TX)^{-1} X^T y$

In [ ]:
gMSE_all = []
nMSE_all = []
i_all = []

theta = np.random.randn(X_train.shape[1]) * np.sqrt(2)
theta_hat = np.linalg.pinv(X_train.T.dot(X_train)).dot(X_train.T).dot(y_train)

alpha = 0.05

for i in range(1000):
    y_hat_gd = X_train @ theta
    gMSE = (1 / X_train.shape[0]) * np.sum((y_hat_gd - y_train) ** 2)
    gMSE_all.append(gMSE)
    i_all.append(i)

    delta = (2 / X_train.shape[0]) * (X_train.T @ (y_hat_gd - y_train))
    theta = theta - alpha * delta

    y_hat_ne = X_train @ theta_hat
    nMSE = (1 / X_train.shape[0]) * np.sum((y_hat_ne - y_train) ** 2)
    nMSE_all.append(nMSE)

gy_hat = X_test @ theta
ny_hat = X_test @ theta_hat

gMSE_test = (1 / X_test.shape[0]) * np.sum((gy_hat - y_test) ** 2)
nMSE_test = (1 / X_test.shape[0]) * np.sum((ny_hat - y_test) ** 2)

fig, ax = plt.subplots(1, 2, figsize=(20, 5))

ax[0].plot(i_all[:200], gMSE_all[:200], label="GD")
ax[0].plot(i_all[:200], nMSE_all[:200], label="Normal Eq")
ax[0].set_xlabel("Iteration")
ax[0].set_ylabel("MSE")
ax[0].legend()

ax[1].scatter(range(X_test.shape[0]), gy_hat - y_test, label="GD", s=5, alpha=0.3)
ax[1].scatter(range(X_test.shape[0]), ny_hat - y_test, label="Normal Eq", s=5, alpha=0.3, c='goldenrod')
ax[1].set_xlabel("Sample")
ax[1].set_ylabel("Error ($)")
ax[1].legend()

text = f"Test MSE:\nGD: {round(gMSE_test, 2)}\nNormal Eq: {round(nMSE_test, 2)}"
ax[1].text(0.02, 0.95, text, transform=ax[1].transAxes, verticalalignment='top',
           fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## Predicted vs actual

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=1)
X_test_1d = pca.fit_transform(X_test[:, 1:])

sort_idx = X_test_1d.flatten().argsort()

fig, ax = plt.subplots(figsize=(14, 6))
ax.scatter(X_test_1d[sort_idx], y_test[sort_idx], alpha=0.2, s=8, label='Actual', color='steelblue')
ax.scatter(X_test_1d[sort_idx], ny_hat[sort_idx], alpha=0.2, s=8, label='Predicted', color='coral')
ax.set_xlabel('PCA Component 1')
ax.set_ylabel('Median House Value ($)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
plot_features = ['median_income', 'housing_median_age', 'latitude', 'longitude']
plot_indices = [feature_names.index(f) + 1 for f in plot_features]

fig, axs = plt.subplots(2, 2, figsize=(20, 16))

for idx, (feat, col_idx) in enumerate(zip(plot_features, plot_indices)):
    row, col = idx // 2, idx % 2
    axs[row, col].scatter(X_test[:, col_idx], y_test, alpha=0.15, s=5, label='Actual')
    axs[row, col].scatter(X_test[:, col_idx], ny_hat, alpha=0.15, s=5, label='Predicted', color='coral')
    axs[row, col].set_xlabel(feat)
    axs[row, col].set_ylabel('median_house_value')
    axs[row, col].legend()

plt.tight_layout()
plt.show()

In [ ]:
long_idx = feature_names.index('longitude') + 1
lat_idx = feature_names.index('latitude') + 1

fig = plt.figure(figsize=(16, 10))
ax = fig.add_subplot(projection='3d')

ax.scatter3D(X_test[:, long_idx], X_test[:, lat_idx], ny_hat,
             c=ny_hat, cmap='Reds', alpha=0.3, s=5, label='Predicted')
ax.scatter3D(X_test[:, long_idx], X_test[:, lat_idx], y_test,
             c=y_test, cmap='Greens', alpha=0.3, s=5, label='Actual')

ax.set_xlabel('Longitude (z)')
ax.set_ylabel('Latitude (z)')
ax.set_zlabel('Median House Value ($)')
ax.legend()
ax.view_init(20, 30)
plt.tight_layout()
plt.show()

## Metrics

In [ ]:
y_hat = X_test @ theta_hat

MSE = (1 / X_test.shape[0]) * np.sum((y_hat - y_test) ** 2)
MAE = (1 / X_test.shape[0]) * np.sum(np.abs(y_hat - y_test))
R2 = sk.metrics.r2_score(y_test, y_hat)

print(f"MSE: {np.round(MSE, 2)}")
print(f"MAE: {np.round(MAE, 2)}")
print(f"R2:  {np.round(R2, 4)}")

## Ridge, LASSO, Elastic Net

In [ ]:
ridge_reg = linear_model.Ridge(alpha=1, solver='cholesky')
ridge_reg.fit(X_train, y_train)
y_hat_ridge = ridge_reg.predict(X_test)

lasso_reg = linear_model.Lasso(alpha=100)
lasso_reg.fit(X_train, y_train)
y_hat_lasso = lasso_reg.predict(X_test)

elastic_net = linear_model.ElasticNet(alpha=1, l1_ratio=0.5)
elastic_net.fit(X_train, y_train)
y_hat_elastic = elastic_net.predict(X_test)

all_predictions = [y_hat, y_hat_ridge, y_hat_lasso, y_hat_elastic]
labels = ['Linear', 'Ridge', 'LASSO', 'Elastic Net']

fig, ax = plt.subplots(4, 1, figsize=(18, 28))

for i, (pred, label) in enumerate(zip(all_predictions, labels)):
    ax[i].scatter(range(X_test.shape[0]), pred, s=5, alpha=0.3, label='Predicted')
    ax[i].scatter(range(X_test.shape[0]), y_test, s=5, alpha=0.3, label='Actual')
    ax[i].set_title(label)
    ax[i].set_xlabel('Sample')
    ax[i].set_ylabel('House Value ($)')
    ax[i].legend()

    mse = np.round((1 / X_test.shape[0]) * np.sum((pred - y_test) ** 2), 2)
    mae = np.round((1 / X_test.shape[0]) * np.sum(np.abs(pred - y_test)), 2)
    r2 = np.round(sk.metrics.r2_score(y_test, pred), 4)
    errors = f"MSE: {mse}\nMAE: {mae}\nR2: {r2}"
    ax[i].text(0.01, 0.95, errors, transform=ax[i].transAxes, verticalalignment='top',
              fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()